In [33]:
import os
from pathlib import Path
from typing import Dict
import numpy as np
import math

import torch
from torch import nn
from torch.nn import functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchvision.utils import save_image
from torch_fidelity import calculate_metrics


In [21]:
def create_experiment(folder_name, base_path="."):
    """Creates a new folder for the experiment. If a folder with the same name already exists, it appends a number to the folder name."""
    
    base_path = Path(base_path)
    folder_path = base_path / folder_name

    counter = 1

    while folder_path.exists():
        folder_path = base_path / f"{folder_name}_{counter}"
        counter += 1

    folder_path.mkdir(parents=True, exist_ok=False)
    os.makedirs(folder_path/"CheckpointsCondition" , exist_ok=False)
    os.makedirs(folder_path/"SampledImgs" , exist_ok=False)

    return str(folder_path)

experiment_dir = create_experiment("new_version")

In [3]:
def extract(v, t, x_shape):
    """
    Extract some coefficients at specified timesteps, then reshape to
    [batch_size, 1, 1, 1, 1, ...] for broadcasting purposes.
    """
    device = t.device
    out = torch.gather(v, index=t, dim=0).float().to(device)
    return out.view([t.shape[0]] + [1] * (len(x_shape) - 1))

Here I want to clarify `coeff1` and `coeff1`
$$
\mu_t = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta \right)
$$


$$
\text{coeff}_1 =\frac{1}{\sqrt{\alpha_t}} 
$$


$$
\text{coeff}_2 =\frac{1}{\sqrt{\alpha_t}}  * \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}
$$

In [ ]:
class Diffusion(nn.Module):
    def __init__(self, model, beta_1, beta_T, T, w=0.0):
        super().__init__()

        self.model = model
        self.T = T

        # CFG strength
        self.w = w

        # Noise schedule / Linear schedule
        self.register_buffer(
            "betas",
            torch.linspace(beta_1, beta_T, T).double()
        )
        
        alphas = 1.0 - self.betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        alphas_bar_prev = F.pad(alphas_bar, [1, 0], value=1)[:T]

        # Forward process terms
        self.register_buffer(
            "sqrt_alphas_bar",
            torch.sqrt(alphas_bar)
        )

        self.register_buffer(
            "sqrt_one_minus_alphas_bar",
            torch.sqrt(1.0 - alphas_bar)
        )

        # Reverse process terms
        self.register_buffer(
            "coeff1",
            torch.sqrt(1.0 / alphas)
        )

        self.register_buffer(
            "coeff2",
            torch.sqrt(1.0 / alphas) *
            (1.0 - alphas) / torch.sqrt(1.0 - alphas_bar)
        )

        self.register_buffer(
            "posterior_var",
            self.betas * (1.0 - alphas_bar_prev) / (1.0 - alphas_bar)
        )

    # =========================================================
    # q(x_t | x_0)
    # =========================================================
    def q_sample(self, x_0, t, noise=None):
        """
        Forward diffusion process:
        q(x_t | x_0)
        """

        if noise is None:
            noise = torch.randn_like(x_0)

        x_t = (
            extract(self.sqrt_alphas_bar, t, x_0.shape) * x_0 +
            extract(self.sqrt_one_minus_alphas_bar, t, x_0.shape) * noise
        )

        return x_t, noise

    # =========================================================
    # p(x_{t-1} | x_t)
    # =========================================================
    @torch.no_grad()
    def p_sample(self, x_T, labels):
        """
        Reverse diffusion sampling process.
        """

        x_t = x_T

        for time_step in reversed(range(self.T)):
            
            # Create a tensor of the current time step for the batch
            t = torch.full(
                (x_t.shape[0],),
                time_step,
                device=x_t.device,
                dtype=torch.long
            )

            # Predict conditional noise
            eps_cond = self.model(x_t, t, labels)

            # ================= CFG =================
            # Predict unconditional noise
            eps_uncond = self.model(
                x_t,
                t,
                torch.zeros_like(labels).to(labels.device) # Unconditional embedding (all zeros)
            )

            # CFG formula
            eps = (1.0 + self.w) * eps_cond - self.w * eps_uncond
            # =======================================

            # Mean prediction
            mean = (
                extract(self.coeff1, t, x_t.shape) * x_t -
                extract(self.coeff2, t, x_t.shape) * eps
            )

            # Adapted from Here : https://huggingface.co/blog/annotated-diffusion
            # In [Ho et al., 2020], the authors set Σθ​(xt​,t)=σt2​I where σt is not learned. They found that fixing σ^2t to the variance of the forward process, i.e. σt2​=βt​, worked well in practice. We follow this design choice and set the variance to βt​I.
            var = extract(self.posterior_var, t, x_t.shape)

            # Add noise except at final step
            if time_step > 0:
                noise = torch.randn_like(x_t)
            else:
                noise = 0

            x_t = mean + torch.sqrt(var) * noise

            assert torch.isnan(x_t).sum() == 0, "nan in tensor."

        return torch.clamp(x_t, -1, 1)

    # =========================================================
    # Training loss
    # =========================================================
    def loss(self, x_0, labels):
        """
        DDPM training objective.
        """

        t = torch.randint(
            0,
            self.T,
            (x_0.shape[0],),
            device=x_0.device
        )

        noise = torch.randn_like(x_0)

        x_t, noise = self.q_sample(x_0, t, noise) # Return noise if we havn't provided it as an argument

        pred_noise = self.model(x_t, t, labels)

        loss = F.mse_loss(pred_noise, noise)

        return loss

In [5]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, T, d_model, dim):
        assert d_model % 2 == 0
        super().__init__()

        emb = (torch.arange(0, d_model, step=2) / d_model) * math.log(10000) 
        emb = torch.exp(-emb)
        pos = torch.arange(T).float()

        # Outer Product to get the full T x d_model//2 matrix
        emb = pos[:, None] * emb[None, :]

        # ensure shape is [T, d_model // 2]
        assert list(emb.shape) == [T, d_model // 2]

        # Here we make odd and even dimensions of the embedding correspond to cos and sin functions respectively
        emb = torch.stack([torch.sin(emb), torch.cos(emb)], dim=-1)

        # ensure shape is [T, d_model // 2, 2]
        assert list(emb.shape) == [T, d_model // 2, 2]
        
        # Take the last two dimensions and flatten them to get a [T, d_model] embedding
        emb = emb.view(T, d_model)
        
        self.timembedding = nn.Sequential(
            nn.Embedding.from_pretrained(emb, freeze=False),
            nn.Linear(d_model, dim),
            Swish(),
            nn.Linear(dim, dim),
        )

    def forward(self, t):
        emb = self.timembedding(t)
        return emb # dim shape:  [B, dim]

In [7]:
class ConditionalEmbedding(nn.Module):
    def __init__(self, num_labels, d_model, dim):
        assert d_model % 2 == 0 # Just to be consistent with the time embedding dimension or if we use AdaGN
        super().__init__()
        
        self.condEmbedding = nn.Sequential(
            nn.Embedding(num_embeddings=num_labels + 1, embedding_dim=d_model, padding_idx=0),
            nn.Linear(d_model, dim),
            Swish(),
            nn.Linear(dim, dim),
        )

    def forward(self, t):
        emb = self.condEmbedding(t)
        return emb # dim shape:  [B, dim]

In [8]:
class DownSample(nn.Module):
    """
        Down Sampling with to different kernel size:
        This gives multi-scale features:
            c1 focuses on finer/local details.
            c2 captures broader context.
    """
    def __init__(self, in_ch):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, in_ch, 3, stride=2, padding=1)
        self.c2 = nn.Conv2d(in_ch, in_ch, 5, stride=2, padding=2) 

    def forward(self, x, temb, cemb): # temb and cemb are not used in this simple implementation by using for loop
        x = self.c1(x) + self.c2(x)
        return x


class UpSample(nn.Module):
    """
        Up Sampling with transposed convolution.
        The reverse 
    """
    def __init__(self, in_ch):
        super().__init__()
        self.c = nn.Conv2d(in_ch, in_ch, 3, stride=1, padding=1)
        self.t = nn.ConvTranspose2d(in_ch, in_ch, 5, 2, 2, 1)

    def forward(self, x, temb, cemb): # temb and cemb are not used in this simple implementation by using for loop
        _, _, H, W = x.shape
        x = self.t(x) # Upsample to (H*2, W*2) using transposed convolution
        x = self.c(x) # Convolution to refine features and to mitigate checkerboard artifacts
        return x

In [9]:
class AttnBlock(nn.Module):
    """Self-Attention block for capturing long-range dependencies in the feature maps."""
    def __init__(self, in_ch):
        super().__init__()
        self.group_norm = nn.GroupNorm(32, in_ch)

        self.proj_q = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0) # 1x1 convolution to project to query space
        self.proj_k = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0) # 1x1 convolution to project to key space
        self.proj_v = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0) # 1x1 convolution to project to value space
        
        self.proj = nn.Conv2d(in_ch, in_ch, 1, stride=1, padding=0)

    def forward(self, x):
        B, C, H, W = x.shape

        h = self.group_norm(x)

        q = self.proj_q(h)
        k = self.proj_k(h)
        v = self.proj_v(h)

        q = q.permute(0, 2, 3, 1).view(B, H * W, C) # Reshape to (B, H*W, C) for attention computation
        k = k.view(B, C, H * W) # Reshape to (B, C, H*W) for attention computation
        w = torch.bmm(q, k) * (int(C) ** (-0.5)) # Scaled dot-product attention

        assert list(w.shape) == [B, H * W, H * W]

        w = F.softmax(w, dim=-1) # Softmax to get attention weights
        v = v.permute(0, 2, 3, 1).view(B, H * W, C) # Reshape to (B, H*W, C) for attention computation
        h = torch.bmm(w, v) # Weighted sum of values based on attention weights

        assert list(h.shape) == [B, H * W, C]

        h = h.view(B, H, W, C).permute(0, 3, 1, 2) # Reshape back to (B, C, H, W)
        h = self.proj(h) # Final projection to get the more rich output of the attention block

        return x + h


In [10]:
# class AdaGN(nn.Module):
#     """
#     Adaptive Group Normalization.
#     Predicts (scale, shift) from a combined condition vector, then applies:
#         y = scale * GroupNorm(x) + shift
#     """
#     def __init__(self, cond_dim: int, out_ch: int, num_groups: int = 32):
#         super().__init__()
#         self.gn    = nn.GroupNorm(num_groups, out_ch, affine=False)
#         self.proj  = nn.Linear(cond_dim, out_ch * 2)   # → (scale, shift)
#         nn.init.zeros_(self.proj.weight)                 # start as identity
#         nn.init.ones_(self.proj.bias[:out_ch])           # scale = 1
#         nn.init.zeros_(self.proj.bias[out_ch:])          # shift = 0

#     def forward(self, x, cond):
#         """x: (B,C,H,W)   cond: (B, cond_dim)"""
#         params        = self.proj(cond)                  # (B, 2*C)
#         scale, shift  = params.chunk(2, dim=1)           # each (B, C)
#         x_norm        = self.gn(x)
#         return x_norm * (1 + scale[:, :, None, None]) + shift[:, :, None, None]


# class ResBlock(nn.Module):
#     """
#     Conditioning via AdaGN.
#     The combined embedding  (temb + cemb)  is used as the AdaGN condition.
#     """
#     def __init__(self, in_ch, out_ch, tdim, dropout, use_attn=False):
#         super().__init__()
#         # first norm is standard GroupNorm (before we have a projection to out_ch)
#         self.norm1     = nn.GroupNorm(32, in_ch)
#         self.act1      = Swish()
#         self.conv1     = nn.Conv2d(in_ch, out_ch, 3, padding=1)

#         # time  embedding maps tdim → tdim  (keeps combined emb at tdim)
#         self.temb_proj = nn.Sequential(Swish(), nn.Linear(tdim, tdim))

#         # AdaGN on the second norm using the combined embedding
#         self.adagn     = AdaGN(cond_dim=tdim, out_ch=out_ch)
#         self.act2      = Swish()
#         self.drop      = nn.Dropout(dropout)
#         self.conv2     = nn.Conv2d(out_ch, out_ch, 3, padding=1)

#         self.shortcut  = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
#         self.attn      = AttnBlock(out_ch) if use_attn else nn.Identity()

#         for m in self.modules():
#             if isinstance(m, (nn.Conv2d, nn.Linear)):
#                 nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
#         nn.init.xavier_uniform_(self.conv2.weight, gain=1e-5)

#     def forward(self, x, temb, cemb):
#         cond = self.temb_proj(temb) + cemb     # (B, tdim) combined condition
#         h    = self.conv1(self.act1(self.norm1(x)))
#         h    = self.adagn(h, cond)             # scale+shift norm
#         h    = self.conv2(self.drop(self.act2(h))) + self.shortcut(x)
#         return self.attn(h)

In [11]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, dropout, use_attn=True):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.GroupNorm(32, in_ch),
            Swish(),
            nn.Conv2d(in_ch, out_ch, 3, stride=1, padding=1),
        )
        self.temb_proj = nn.Sequential(
            Swish(),
            nn.Linear(tdim, out_ch),
        )

        self.cond_proj = nn.Sequential(
            Swish(),
            nn.Linear(tdim, out_ch),
        )
        
        self.block2 = nn.Sequential(
            nn.GroupNorm(32, out_ch),
            Swish(),
            nn.Dropout(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1),
        )

        
        if in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride=1, padding=0)
        else:
            self.shortcut = nn.Identity()


        if use_attn:
            self.attn = AttnBlock(out_ch)
        else:
            self.attn = nn.Identity()


    def forward(self, x, temb, labels):
        h = self.block1(x)

        h += self.temb_proj(temb)[:, :, None, None]
        h += self.cond_proj(labels)[:, :, None, None]

        h = self.block2(h)

        h = h + self.shortcut(x) # Residual connection
        h = self.attn(h) # Attention block.
        return h

In [12]:
class UNet(nn.Module):
    def __init__(self, T, num_labels, ch, ch_mult, num_res_blocks, dropout):
        super().__init__()

        tdim = ch * 4

        self.time_embedding = TimeEmbedding(T, ch, tdim)

        self.cond_embedding = ConditionalEmbedding(num_labels, ch, tdim)

        self.head = nn.Conv2d(3, ch, kernel_size=3, stride=1, padding=1)
        
        self.downblocks = nn.ModuleList()

        chs = [ch]  # record output channel when dowmsample for upsample

        now_ch = ch

        for i, mult in enumerate(ch_mult):

            out_ch = ch * mult

            for _ in range(num_res_blocks):

                self.downblocks.append(ResBlock(in_ch=now_ch, out_ch=out_ch, tdim=tdim, dropout=dropout))

                now_ch = out_ch

                chs.append(now_ch)

            if i != len(ch_mult) - 1:
                self.downblocks.append(DownSample(now_ch))
                chs.append(now_ch)

        self.middleblocks = nn.ModuleList([
            ResBlock(now_ch, now_ch, tdim, dropout, use_attn=True),
            ResBlock(now_ch, now_ch, tdim, dropout, use_attn=False),
        ])

        self.upblocks = nn.ModuleList()
        
        for i, mult in reversed(list(enumerate(ch_mult))):
            out_ch = ch * mult
            for _ in range(num_res_blocks + 1):
                self.upblocks.append(ResBlock(in_ch=chs.pop() + now_ch, out_ch=out_ch, tdim=tdim, dropout=dropout, use_attn=False))
                now_ch = out_ch
            if i != 0:
                self.upblocks.append(UpSample(now_ch))

        self.tail = nn.Sequential(
            nn.GroupNorm(32, now_ch),
            Swish(),
            nn.Conv2d(now_ch, 3, 3, stride=1, padding=1)
        )
 

    def forward(self, x, t, labels):
        # Timestep embedding
        temb = self.time_embedding(t)
        cemb = self.cond_embedding(labels)

        # Downsampling
        h = self.head(x)
        hs = [h]
        for layer in self.downblocks:
            h = layer(h, temb, cemb)
            hs.append(h)

        # Middle
        for layer in self.middleblocks:
            h = layer(h, temb, cemb)
        
        # Upsampling
        for layer in self.upblocks:
            if isinstance(layer, ResBlock):
                h = torch.cat([h, hs.pop()], dim=1)
            h = layer(h, temb, cemb)
        h = self.tail(h)

        assert len(hs) == 0
        return h


Our Classes

| Class        | Label |
|--------------|-------|
| airplane     | 0     |
| automobile   | 1     |
| bird         | 2     |
| cat          | 3     |
| deer         | 4     |
| dog          | 5     |
| frog         | 6     |
| horse        | 7     |
| ship         | 8     |
| truck        | 9     |


In [13]:
def train(modelConfig: Dict):

    device = torch.device(modelConfig["device"])
    # dataset
    dataset = CIFAR10(
        root='./data', train=True, download=True,
        transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]))
    
    dataloader = DataLoader(
        dataset, batch_size=modelConfig["batch_size"], shuffle=True, num_workers=4, drop_last=True, pin_memory=True)

    # model setup
    net_model = UNet(T=modelConfig["T"],
                     num_labels=10,
                     ch=modelConfig["channel"],
                     ch_mult=modelConfig["channel_mult"],
                     num_res_blocks=modelConfig["num_res_blocks"],
                     dropout=modelConfig["dropout"]).to(device)
    
    
    if modelConfig["training_load_weight"] is not None:
        net_model.load_state_dict(torch.load(os.path.join(
            modelConfig["save_dir"],
            modelConfig["training_load_weight"]),
            map_location=device),
            strict=False)
        
        print("Model weight load down.")
    

    optimizer = torch.optim.AdamW(
        net_model.parameters(),
        lr=modelConfig["lr"],
        weight_decay=1e-4)
    
    cosineScheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer=optimizer,
        T_max=modelConfig["epoch"],
        eta_min=0, last_epoch=-1)
    
    trainer = Diffusion(
        net_model, modelConfig["beta_1"], modelConfig["beta_T"], modelConfig["T"]).to(device)

    # start training
    for e in range(modelConfig["epoch"]):
        with tqdm(dataloader, dynamic_ncols=True) as tqdmDataLoader:
            for images, labels in tqdmDataLoader:
                # train
                b = images.shape[0]
                optimizer.zero_grad()
                x_0 = images.to(device)
                labels = labels.to(device) + 1 # Shift labels to start from 1, reserve 0 for uncondtional embedding
                
                # Train with 10% unconditional embedding to improve sample quality as per classifier-free guidance (CFG) paper.
                if np.random.rand() < 0.1:
                    labels = torch.zeros_like(labels).to(device)
                
                loss = trainer.loss(x_0, labels)

                loss.backward()

                torch.nn.utils.clip_grad_norm_(net_model.parameters(), modelConfig["grad_clip"])
                
                optimizer.step()

                tqdmDataLoader.set_postfix(ordered_dict={
                    "epoch": e,
                    "loss: ": loss.item(),
                    "img shape: ": x_0.shape,
                    "LR": optimizer.state_dict()['param_groups'][0]["lr"]
                })

        cosineScheduler.step()

        torch.save(net_model.state_dict(), os.path.join(
            modelConfig["save_dir"], 'ckpt_' + str(e) + "_.pt"))


In [18]:
def generate_samples(modelConfig: Dict):

    device = torch.device(modelConfig["device"])
    # load model and evaluate

    samples_size = 80

    with torch.no_grad():
        step = int(samples_size // 10)

        labelList = []

        k = 0

        # Generate labels in batches of 'step' for each class (0-9)
        for i in range(1, samples_size + 1):

            labelList.append(torch.ones(size=[1]).long() * k) # Generate labels in batches of 10 for each class (0-9)

            # Update label every 'step' samples to ensure we have 10 samples per class
            if i % step == 0:
                if k < 10 - 1:
                    k += 1
        # Concatenate all label batches into a single tensor and shift labels to start from 1 (0 reserved for unconditional embedding)
        labels = torch.cat(labelList, dim=0).long().to(device) + 1

        print("labels: ", labels)

        # Create model and load weights
        model = UNet(
            T=modelConfig["T"],
            num_labels=10,
            ch=modelConfig["channel"],
            ch_mult=modelConfig["channel_mult"],
            num_res_blocks=modelConfig["num_res_blocks"],
            dropout=modelConfig["dropout"]).to(device)
        
        # Load the trained model weights
        ckpt = torch.load(os.path.join(
            modelConfig["save_dir"],
            modelConfig["test_load_weight"]),
            map_location=device)
        
        model.load_state_dict(ckpt)

        print("model load weight done.")

        # Start sampling
        model.eval()
        sampler = Diffusion(
            model, modelConfig["beta_1"], modelConfig["beta_T"], modelConfig["T"], w=modelConfig["w"]).to(device)
        
        # Sampled from standard normal distribution
        noisyImage = torch.randn(
            size=[samples_size, 3, modelConfig["img_size"], modelConfig["img_size"]], device=device)
        
        # Save the initial noisy image for reference (scaled to [0, 1] range for visualization)
        saveNoisy = torch.clamp(noisyImage * 0.5 + 0.5, 0, 1)
        
        save_image(saveNoisy, os.path.join(
            modelConfig["sampled_dir"],  modelConfig["sampledNoisyImgName"]), nrow=modelConfig["nrow"])

        # Generate samples using the reverse diffusion process conditioned on the labels
        sampledImgs = sampler.p_sample(noisyImage, labels)

        sampledImgs = sampledImgs * 0.5 + 0.5  # [0 ~ 1]
        
        print(sampledImgs)

        save_image(sampledImgs, os.path.join(
            modelConfig["sampled_dir"],  modelConfig["sampledImgName"]), nrow=modelConfig["nrow"])

In [24]:
modelConfig = {
    "epoch": 70,
    "batch_size": 128,
    "T": 500,
    "channel": 128,
    "channel_mult": [1, 2, 4],
    "num_res_blocks": 2,
    "dropout": 0.15,
    "lr": 1e-4,
    "multiplier": 2.5,
    "beta_1": 1e-4,
    "beta_T": 0.02,
    "img_size": 32,
    "grad_clip": 1.,
    "device": "cuda",
    "w": 1.8,
    "save_dir": f"{experiment_dir}/CheckpointsCondition/",
    "training_load_weight": None,
    "test_load_weight": "ckpt_69_.pt",
    "sampled_dir": f"{experiment_dir}/SampledImgs/",
    "sampledNoisyImgName": f"NoisyGuidenceImgs.png",
    "sampledImgName": f"SampledGuidenceImgs.png",
    "nrow": 8
}


In [25]:
train(modelConfig)

100%|██████████| 390/390 [03:45<00:00,  1.73it/s, epoch=69, loss: =0.0407, img shape: =torch.Size([128, 3, 32, 32]), LR=5.03e-8]


In [26]:
generate_samples(modelConfig)

labels:  tensor([ 1,  1,  1,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  2,  2,  2,  3,  3,
         3,  3,  3,  3,  3,  3,  4,  4,  4,  4,  4,  4,  4,  4,  5,  5,  5,  5,
         5,  5,  5,  5,  6,  6,  6,  6,  6,  6,  6,  6,  7,  7,  7,  7,  7,  7,
         7,  7,  8,  8,  8,  8,  8,  8,  8,  8,  9,  9,  9,  9,  9,  9,  9,  9,
        10, 10, 10, 10, 10, 10, 10, 10], device='cuda:0')
model load weight done.
tensor([[[[0.4636, 0.4619, 0.4663,  ..., 0.4904, 0.4890, 0.4881],
          [0.4691, 0.4634, 0.4675,  ..., 0.4903, 0.4875, 0.4920],
          [0.4711, 0.4657, 0.4679,  ..., 0.4875, 0.4841, 0.4886],
          ...,
          [0.5641, 0.5564, 0.5586,  ..., 0.5656, 0.5595, 0.5627],
          [0.5779, 0.5693, 0.5694,  ..., 0.5821, 0.5832, 0.5858],
          [0.5986, 0.5846, 0.5826,  ..., 0.5958, 0.6009, 0.5955]],

         [[0.5885, 0.5842, 0.5873,  ..., 0.6161, 0.6152, 0.6185],
          [0.5917, 0.5851, 0.5892,  ..., 0.6170, 0.6169, 0.6199],
          [0.5905, 0.5854, 0.5884,  ..., 0.6

# Evaluation Metrics

In [27]:
def sample_images(
    sampler,
    device,
    img_size=32,
    num_samples=5000,
    batch_size=128,
    num_classes=10
):

    fake_images = []

    generated = 0

    while generated < num_samples:

        current_bs = min(batch_size, num_samples - generated)

        labels = torch.randint(
            0,
            num_classes,
            (current_bs,),
            device=device
        )

        noise = torch.randn(
            current_bs,
            3,
            img_size,
            img_size,
            device=device
        )

        with torch.no_grad():
            samples = sampler.p_sample(noise, labels)

        # [-1,1] -> [0,1]
        samples = (samples * 0.5 + 0.5).clamp(0, 1)

        fake_images.append(samples.cpu())

        generated += current_bs

        print(f"Generated {generated}/{num_samples}")

    fake_images = torch.cat(fake_images, dim=0)

    return fake_images

In [28]:
def save_generated_images(fake_imgs, save_dir="generated_images"):

    os.makedirs(save_dir, exist_ok=True)

    for i, img in enumerate(fake_imgs):

        save_image(
            img,
            os.path.join(save_dir, f"{i:05d}.png")
        )

    print(f"Saved {len(fake_imgs)} images to {save_dir}")

In [29]:
def load_model_and_sampler(modelConfig, device):

    model = UNet(
        T=modelConfig["T"],
        num_labels=10,
        ch=modelConfig["channel"],
        ch_mult=modelConfig["channel_mult"],
        num_res_blocks=modelConfig["num_res_blocks"],
        dropout=modelConfig["dropout"]
    ).to(device)

    ckpt = torch.load(
        os.path.join(
            modelConfig["save_dir"],
            modelConfig["test_load_weight"]
        ),
        map_location=device
    )

    model.load_state_dict(ckpt)

    model.eval()

    sampler = Diffusion(
        model,
        modelConfig["beta_1"],
        modelConfig["beta_T"],
        modelConfig["T"],
        w=modelConfig["w"]
    ).to(device)

    return model, sampler

In [30]:

def compute_fid_is(fake_dir):

    metrics = calculate_metrics(
        input1=fake_dir,
        input2="cifar10-train",
        cuda=torch.cuda.is_available(),
        fid=True,
        isc=True,
        verbose=False
    )

    return {
        "FID": metrics["frechet_inception_distance"],
        "IS": metrics["inception_score_mean"]
    }

In [31]:
def eval_model(modelConfig):

    device = torch.device(modelConfig["device"])

    print("Loading model...")
    _, sampler = load_model_and_sampler(modelConfig, device)

    print("Generating fake images...")
    fake_imgs = sample_images(
        sampler=sampler,
        device=device,
        img_size=32,
        num_samples=5000,
        batch_size=128
    )

    # =======================================
    # SAVE GENERATED IMAGES
    # =======================================
    fake_dir = f"{experiment_dir}/generated_eval_images"

    save_generated_images(
        fake_imgs,
        save_dir=fake_dir
    )

    # =======================================
    # FID + IS
    # =======================================
    print("Computing FID and IS...")

    fid_is_metrics = compute_fid_is(fake_dir)

    print("\n========== RESULTS ==========")

    for k, v in fid_is_metrics.items():
        print(f"{k}: {v:.4f}")

In [34]:
fid_is_metrics = compute_fid_is(f"{experiment_dir}/generated_eval_images")

print("\n========== RESULTS ==========")

for k, v in fid_is_metrics.items():
    print(f"{k}: {v:.4f}")


========== RESULTS ==========
FID: 17.2369
IS: 8.6970


In [32]:
eval_model(modelConfig)

Loading model...
Generating fake images...
Generated 128/5000
Generated 256/5000
Generated 384/5000
Generated 512/5000
Generated 640/5000
Generated 768/5000
Generated 896/5000
Generated 1024/5000
Generated 1152/5000
Generated 1280/5000
Generated 1408/5000
Generated 1536/5000
Generated 1664/5000
Generated 1792/5000
Generated 1920/5000
Generated 2048/5000
Generated 2176/5000
Generated 2304/5000
Generated 2432/5000
Generated 2560/5000
Generated 2688/5000
Generated 2816/5000
Generated 2944/5000
Generated 3072/5000
Generated 3200/5000
Generated 3328/5000
Generated 3456/5000
Generated 3584/5000
Generated 3712/5000
Generated 3840/5000
Generated 3968/5000
Generated 4096/5000
Generated 4224/5000
Generated 4352/5000
Generated 4480/5000
Generated 4608/5000
Generated 4736/5000
Generated 4864/5000
Generated 4992/5000
Generated 5000/5000
Saved 5000 images to new_version/generated_eval_images
Computing FID and IS...


NameError: name 'calculate_metrics' is not defined